# Annotation of NK13

In [ ]:
import anndata as ad
import scanpy as sc
import scanpy.external as sce
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import os
import muon as mu
from scipy.stats import binomtest
from scipy.stats import mannwhitneyu
import string

# Helps to define plotting area
import matplotlib.pyplot as plt
from matplotlib import rcParams 
from matplotlib.pyplot import rc_context
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib import colors
# from matplotlib_venn import venn2
from scipy.stats import spearmanr
import matplotlib.patches as mpatches

from pathlib import Path

## For making text editable in .pdfs once exported use cairo backend instead of agg
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# set seed
np.random.seed(44)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999

# Loading data

In [ ]:
mdata = mu.read(filename="../../04_data_objects/06_preprocessed_v2/2026_6_multimer_dataset_preprocessed.h5mu")

In [ ]:
mdata["gex"].obs[["chain_pairing", "donor", "clone_handle", "antigen.species", "antigen.gene", "antigen.epitope", "specific"]]

# Preprocess

## Annotate mdata.obs with epitope and HLA information

In [ ]:
epitope_ref = pd.read_excel("../../03_annotation/11_multimer_experiment_ann/FeatureBarcodCorrected.xlsx")
epitope_ref = epitope_ref.dropna()

# Select colums to mergee
epitope_ref = epitope_ref[["id", "HLA", "Peptide", "Sequence", "Virus"]]
epitope_ref = epitope_ref.rename(columns={"id":"specific"})

#Annotate mdata["gex"].obs

# Create dictionary mapping 'specific' -> dict of features
specific_dict = epitope_ref.set_index('specific')[['HLA', 'Peptide', 'Sequence', 'Virus']].T.to_dict()

# Map the dictionary values to new columns in the .obs object. 
for feature in ['HLA', 'Peptide', 'Sequence', 'Virus']:
    mdata["gex"].obs[feature] = mdata["gex"].obs['specific'].map(lambda x: specific_dict.get(x, {}).get(feature))

mdata["gex"].obs.columns

In [ ]:
mdata["gex"].obs["HLA"].drop_duplicates().tolist()

In [ ]:
## Check 
mdata["gex"].obs[["chain_pairing", "donor", "clone_handle", "antigen.species", "antigen.gene", "antigen.epitope", "specific", "HLA", "Peptide", "Sequence", "Virus"]]

## Annotate handle

In [ ]:
mdata["gex"].obs["handle"] = mdata["gex"].obs["Sequence"] + "_" + mdata["gex"].obs["HLA"] + "_" + mdata["gex"].obs["Virus"] 

## Split Pt17/ Pt21 cells
An error occured when handling hashtag-antibodies in the lab. We correct it here turning 3374 cells from "Pt17/Pt21" into only 1643 cells that cannot be assigned

In [ ]:
mdata["gex"].obs["donor"].value_counts()

In [ ]:
mdata["gex"].obs["donor"] = pd.Categorical(np.where(
    (mdata["gex"].obs["donor"] == "Pt17/Pt21") & (mdata["gex"].obs["HLA"] == "A0201"), "Pt21", np.where(
         (mdata["gex"].obs["donor"] == "Pt17/Pt21") & (mdata["gex"].obs["HLA"].isna()), "Pt17/Pt21", np.where(
             (mdata["gex"].obs["donor"] == "Pt17/Pt21"), "Pt17", mdata["gex"].obs["donor"]))))

In [ ]:
## Check 
mdata["gex"].obs[["chain_pairing", "donor", "clone_handle", "antigen.species", "antigen.gene", "antigen.epitope", "specific", "HLA", "Peptide", "Sequence", "Virus"]].head(2)

In [ ]:
## Check Pt17 specifically
mdata["gex"].obs[mdata["gex"].obs["donor"] == "Pt17"][["chain_pairing", "donor", "clone_handle", "clone_id", "antigen.species", "antigen.gene", "antigen.epitope", "specific", "HLA", "Peptide", "Sequence", "Virus"]].head(2)

# Annotate catpat

In [ ]:
#Group annotation
patient_list = ["Pt4", "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt5", "Pt8", "Pt18", "Pt19", "Pt21"]

#Match the lists defined above to define "RRMS" and "Control"
mdata["gex"].obs["catpat"] = pd.Categorical(np.where(mdata["gex"].obs["donor"].isin(patient_list), "RRMS", np.where(mdata["gex"].obs["donor"].isin(control_list), "Control", "Error")))


In [ ]:
mdata["gex"].obs["donor"].value_counts()

## Update clone_handle and clone_size
clone_id was calculated regardless of donor ID - so the same public clone has the same number in two different donors. Though expansion should be calculated according to clone_handle

In [ ]:
mdata["gex"].obs["clone_handle"] = mdata["gex"].obs["donor"].astype(str) + "-" + mdata["gex"].obs["clone_id"].astype(str) 

## Calculate clone_size = number of rows per clone_id
mdata["gex"].obs["clone_size"] = mdata["gex"].obs.groupby("clone_handle")["clone_handle"].transform("count")

## Set clone_size to 0 if the clone_id is nan
mdata["gex"].obs["clone_size"] = np.where(mdata["gex"].obs["clone_id"].isna(), 0, mdata["gex"].obs["clone_size"])

## Share labels with airr
mdata["airr"].obs["clone_size"] = mdata["gex"].obs["clone_size"].copy()
mdata["prot"].obs["clone_size"] = mdata["gex"].obs["clone_size"].copy()

In [ ]:
## Check 
mdata["gex"].obs[["chain_pairing", "donor", "clone_handle", "antigen.species", "clone_size", "antigen.gene", "antigen.epitope", "specific", "HLA", "Peptide", "Sequence", "Virus"]].sort_values("clone_size", ascending=False).head(2)

## Annotate each specificity with whether or not they are HLA matched. 

In [ ]:
# HLA list
collapsed = pd.read_csv("../../03_annotation/10_metadata_patients/HLA_table.csv")

#columns that is not SampleName = HLA columns
hla_cols = collapsed.columns.difference(["SampleName"])

#Removes * and : special characters
collapsed[hla_cols] = (
    collapsed[hla_cols]
    .astype(str)
    .replace(r"[\*\:]", "", regex=True)
)

# HLA columns collapsed into a list
collapsed = (
    collapsed
    .melt(id_vars="SampleName", value_vars=hla_cols)
    .groupby("SampleName")["value"]
    .apply(list)
    .reset_index(name="HLA_collapsed")
)

#Rename sample name and convert the sample name in mdata to string
collapsed = collapsed.rename(columns={"SampleName": "donor"})
mdata["gex"].obs["donor"] = mdata["gex"].obs["donor"].astype(str)

# ------------------------
# Create dictionary mapping 'specific' -> dict of features
# ------------------------
specific_dict = collapsed.set_index('donor')[['HLA_collapsed']].T.to_dict()

# Map the dictionary values to new columns in the .obs object. 
for feature in ['HLA_collapsed']:
    mdata["gex"].obs[feature] = mdata["gex"].obs['donor'].map(lambda x: specific_dict.get(x, {}).get(feature))

# Convert to string and remove special characters
mdata["gex"].obs["HLA_collapsed"] = mdata["gex"].obs["HLA_collapsed"].astype(str)
mdata["gex"].obs["HLA_collapsed"] = (
    mdata["gex"].obs["HLA_collapsed"]
    .str.replace(r"[\[\]',]", "", regex=True)
    .str.strip()
)


In [ ]:
# Create a series that contains a list of HLAs
hla_lists = mdata["gex"].obs["HLA_collapsed"].astype(str)

#Combine aligns two series by indexes. lambda function then taks lst as the "hla_list" and checks whether hla ("HLA") is in it, and returns matched if it is (missmatched if not)
mdata["gex"].obs["HLAmatch"] = np.where(mdata["gex"].obs["HLA"].astype(str) == "None", "None", 
                                        np.where(hla_lists.combine(mdata["gex"].obs["HLA"].astype(str), lambda lst, hla: hla in lst),
    "HLA_matched",
    "HLA_mismatched"
))

# check
check_df = mdata["gex"].obs[["donor","HLA", "Sequence", "HLA_collapsed", "HLAmatch"]].sort_values("HLAmatch")
check_df[check_df["HLAmatch"] == "HLA_mismatched"].head(5)

## Heatmaps

#### Define a credible clone

In [ ]:
temp_df = mdata["gex"].obs[["chain_pairing", "donor", "clone_handle", "antigen.species", "clone_size", "antigen.gene", "antigen.epitope", "specific", "handle", "HLA", "Peptide", "Sequence", "Virus", "HLA_collapsed", "HLAmatch"]]

# Define clones with more than 0 cells
expanded_list = temp_df[temp_df["clone_size"] > 0][["clone_handle"]].drop_duplicates()["clone_handle"].tolist()
matched_response_list = temp_df[temp_df["HLAmatch"]=="HLA_matched"][["clone_handle"]].drop_duplicates()["clone_handle"].tolist()

In [ ]:
#define credible clones as those with more than 3 cells, and those that are matched to their respective HLA
mdata["gex"].obs["Credible_clones"] = np.where((mdata["gex"].obs["clone_handle"].isin(expanded_list)) & 
                                               (mdata["gex"].obs["clone_handle"].isin(matched_response_list)), "Credible", "Sketchy")

In [ ]:
mdata["gex"].obs["IR_VJ_1_junction_aa"] = mdata["airr"].obs["IR_VJ_1_junction_aa"].copy()
mdata["gex"].obs["IR_VJ_2_junction_aa"] = mdata["airr"].obs["IR_VJ_2_junction_aa"].copy()
mdata["gex"].obs["IR_VDJ_1_junction_aa"] = mdata["airr"].obs["IR_VDJ_1_junction_aa"].copy()

In [ ]:
## So now we need to calculate a mean of all log_sc_10s_138 for all credible clones. 
hm_df = mdata["gex"].obs.copy()

# Manual columns - WILL BE USED TO GROUP!
manual_cols = ["donor","clone_handle", "clone_size", "handle", "HLA"]

# Select columns that contain "log_sc-1OS"
selected_cols = hm_df.columns[hm_df.columns.str.contains("log_sc-1OS")]

# Combine with manual columns --- 
final_cols = list(selected_cols) + manual_cols

# Subset the dataframe
hm_df = hm_df[final_cols]

# summarize and reduce the dataframe to mean values. 
hm_df = hm_df.groupby(manual_cols, as_index=False, observed=True)[list(selected_cols)].mean()

#### Replace column names with multimer names

In [ ]:
epitope_ref = pd.read_excel("../../03_annotation/11_multimer_experiment_ann/FeatureBarcodCorrected.xlsx")
epitope_ref = epitope_ref.dropna()

#Define a column name mirroring the "log_XXX" columns
epitope_ref["targ_col"] = "log_" + epitope_ref["id"]
epitope_ref["handle"] = epitope_ref["Sequence"] + "_" + epitope_ref["HLA"] + "_" + epitope_ref["Virus"] 

# Create dictionary mapping targ_col -->  handle
col_dict = epitope_ref[["targ_col", "handle"]].set_index('targ_col')['handle'].to_dict()
col_dict

# Rename columns where there is a match
hm_df = hm_df.rename(columns=col_dict)

#### For a given HLA select the right multimer columns

In [ ]:
# Select the right rows
hm_subset_df = hm_df[hm_df["HLA"] == "B3501"]

# Select the right columns
manual_cols = ["donor","clone_handle", "clone_size", "handle", "HLA"] # your manually selected columns

# Select columns that contain "log_sc-1OS"
selected_cols = hm_subset_df.columns[hm_subset_df.columns.str.contains("B3501")]

# Combine with manual columns
final_cols =  manual_cols + list(selected_cols)

# Subset the dataframe
hm_subset_df = hm_subset_df[final_cols]
hm_subset_df = hm_subset_df[hm_subset_df["clone_size"] >= 2]

# --------------------------------------------------
# 1) Identify numerical epitope columns
# --------------------------------------------------
value_cols = hm_subset_df.columns[hm_subset_df.columns.str.contains("_B3501_")]

# --------------------------------------------------
# 2) Build the heatmap matrix
#    Rows   -> epitope columns
#    Cols   -> clone_handle
# --------------------------------------------------
heatmap_data = (
    hm_subset_df.set_index("clone_handle")[value_cols]
    .T   # transpose to swap x and y
)

# --------------------------------------------------
# 3) Optional ordering for better readability
#    - order epitopes by virus
#    - order clones by strongest response
# --------------------------------------------------
epitope_order = sorted(heatmap_data.index, key=lambda x: x.split("_")[-1])
clone_order = heatmap_data.max(axis=0).sort_values(ascending=False).index

heatmap_data = heatmap_data.loc[epitope_order, clone_order]

# --------------------------------------------------
# 4) Plot the heatmap
# --------------------------------------------------
plt.figure(figsize=(10, 5))
sb.heatmap(
    heatmap_data,
    cmap="RdBu_r",
    # mask=heatmap_data == 0,
    linewidths=0,
    # linecolor="white"
)

# --------------------------------------------------
# 5) Labels and layout tweaks
# --------------------------------------------------
plt.xlabel("Clone handle")
plt.ylabel("Epitope (peptide_HLA_virus)")
plt.tight_layout()
plt.savefig("../../05_plots/09_multimers/heatmap_B3501.pdf", format="pdf", dpi = 400)
plt.show()

In [ ]:
def plot_hla_heatmap(
    hm_df,
    title_patient, 
    sketchy_flag=0,
    min_clone_size=2,
    figsize=None,
    cmap="RdBu_r",
    savepath=None
):
    """
    Create an epitope × clone heatmap for one or multiple HLAs.

    By default, saves to:
    ../../05_plots/09_multimers/heatmap_<HLA1>_<HLA2>.pdf
    """

    # --------------------------------------------------
    # 0) Prepare the HLA list
    # --------------------------------------------------
    donor_hlas = {
        "Pt1" : ["A0201", "B0702", "C0702"],
        "Pt4" : ["B3501"],
        "Pt5" : ["A0101", "A2402", "B0801", "B1501"],
        "Pt8" : ["A0201", "B0702", "C0702"],
        "Pt11" : ["A0101", "B1501"], 
        "Pt12" : ["A0201"], 
        "Pt14" : ["A0201", "A6801", "B0702", "C0702"], 
        "Pt17" : ["A0101", "B1501", "B3501"], 
        "Pt18" : ["A0101", "A2402", "B0801"], 
        "Pt19" : ["A0201", "A2402", "B0702", "B1501", "C0702"], 
        "Pt20" : ["A0201", "A6801", "B0702", "B0801", "C0702"], 
        "Pt21" : ["A0201", "A2402", "B0702", "C0702"], 
    }

    #HLA below depends on the title patient
    hla = donor_hlas[title_patient]
    
    if isinstance(hla, str):
        hla_list = [hla]
    else:
        hla_list = list(hla)

    hla_tag = "_".join(hla_list)

    # --------------------------------------------------
    # 1) Define default save path
    # --------------------------------------------------
    if savepath is None:
        savepath = (
            Path("../../05_plots/09_multimers")
            / f"heatmap_{hla_tag}_{title_patient}.pdf"
        )

    # --------------------------------------------------
    # 2) Subset rows for selected HLAs
    # --------------------------------------------------
    hm_subset_df = hm_df[hm_df["HLA"].isin(hla_list)].copy()

    # --------------------------------------------------
    # 3) Select relevant columns
    # --------------------------------------------------
    manual_cols = [
        "donor",
        "clone_handle",
        "handle",
        "HLA",
        "clone_size",
    ]

    # Define manual cols again
    manual_cols = ["donor","clone_handle", "clone_size", "handle", "HLA"] # your manually selected columns
    
    # columns that contain ANY of the HLAs
    selected_cols = hm_subset_df.columns[
        hm_subset_df.columns.str.contains("|".join(hla_list))
    ]

    final_cols = manual_cols + list(selected_cols)
    hm_subset_df = hm_subset_df.loc[:, final_cols]

    # Filter on clone size, patient. 
    hm_subset_df = hm_subset_df[
        (hm_subset_df["clone_size"] >= min_clone_size) &
        (hm_subset_df["donor"] == title_patient)
    ]

    # --------------------------------------------------
    # 4) Identify numerical epitope columns
    #    (e.g. peptide_A0101_virus OR peptide_A0201_virus)
    # --------------------------------------------------
    value_cols = hm_subset_df.columns[
        hm_subset_df.columns.str.contains(
            "|".join([f"_{h}_" for h in hla_list])
        )
    ]

    # --------------------------------------------------
    # 5) Build the heatmap matrix
    # --------------------------------------------------
    heatmap_data = (
        hm_subset_df
        .set_index("clone_handle")[value_cols]
        .apply(pd.to_numeric, errors="coerce")
        .groupby(level=0)
        .mean()
        .T
    )
    
    # --------------------------------------------------
    # 6) Ordering
    # --------------------------------------------------
    
    # Sort epitopes per HLA
    epitope_order = sorted(
        heatmap_data.index,
        key=lambda x: x.split("_")[1]  # HLA position
    )
    
    # Sort each clone by the maximum epitope / multimer identity
    # 1) epitope of max response (y-axis identity)
    max_epitope = heatmap_data.idxmax(axis=0)
    
    # 2) value of that max response
    max_value = heatmap_data.max(axis=0)
    
    # Combine into a DataFrame for sorting
    clone_sort_df = pd.DataFrame({
        "clone": heatmap_data.columns,
        "max_epitope": max_epitope,
        "max_value": max_value,
    })
    
    # Sort:
    #   - group by epitope
    #   - within each epitope, sort by response strength
    clone_sort_df = clone_sort_df.sort_values(
        by=["max_epitope", "max_value"],
        ascending=[True, False],
    )
    
    clone_order = clone_sort_df["clone"]
    
    # implement orders. 
    heatmap_data = heatmap_data.loc[epitope_order, clone_order]

    # --------------------------------------------------
    # Dynamic figure size
    # --------------------------------------------------
    n_epitopes, n_clones = heatmap_data.shape
    
    # tunable scaling factors
    width_per_clone = 0.25
    height_per_epitope = 0.2
    
    min_width, max_width = 6, 50
    min_height, max_height = 4, 25
    
    if figsize is None:
        fig_width = min(
            max(n_clones * width_per_clone, min_width),
            max_width)
        
        fig_height = min(
            max(n_epitopes * height_per_epitope, min_height),
            max_height)
        
    figsize = (fig_width, fig_height)

    
    # --------------------------------------------------
    # 7) Plot
    # --------------------------------------------------
    fig, ax = plt.subplots(figsize=figsize)
    
    ax = sb.heatmap(
        heatmap_data,
        cmap=cmap,
        linewidths=0,
    )
    
    plt.draw()  # IMPORTANT for heatmaps

    if sketchy_flag==1: 
        clone_highlight = set(hm_subset_df[hm_subset_df["Credible_clones"] == "Sketchy"]["clone_handle"].drop_duplicates().tolist())
        for label in ax.get_xticklabels():
            if label.get_text() in clone_highlight:
                label.set_color("red")
                label.set_fontweight("bold")
            else:
                label.set_color("black")
        
            label.set_rotation(90)
            label.set_ha("center")    
    else:
        clone_highlight = set([
        "Pt17-3230", "Pt14-3355", "Pt17-3489", "Pt20-3346", "Pt21-3293",
        "Pt21-3549", "Pt14-3271", "Pt14-3299", "Pt14-3423", "Pt14-4688",
        "Pt14-5239", "Pt20-3297", "Pt20-3388", "Pt20-3589", "Pt20-3755",
        "Pt20-3739", "Pt20-4307", "Pt20-4969", "Pt4-87", "Pt4-351",
        "Pt4-2247", "Pt17-4776", "Pt4-923", "Pt4-1055", "Pt4-469",
        "Pt4-515", "Pt14-4523", "Pt17-3351", "Pt5-664", "Pt20-3471",
        "Pt20-5422", "Pt20-5068"
        ])
        for label in ax.get_xticklabels():
            if label.get_text() in clone_highlight:
                label.set_color("red")
                label.set_fontweight("bold")
            else:
                label.set_color("black")
        
            label.set_rotation(90)
            label.set_ha("center")
    
    plt.xlabel("Clone handle")
    plt.ylabel("Epitope (peptide_HLA_virus)")
    plt.title(f"HLA: {', '.join(hla_list)}, {title_patient}")
    plt.tight_layout()

    # --------------------------------------------------
    # 8) Save + show
    # --------------------------------------------------
    savepath.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(savepath, format="pdf", dpi=400)
    plt.show()

### Export heatmaps

In [ ]:
plot_hla_heatmap(hm_df = hm_df, title_patient="Pt1", sketchy_flag=0)

In [ ]:
plot_hla_heatmap(hm_df = hm_df, title_patient="Pt4", sketchy_flag=0)

In [ ]:
plot_hla_heatmap(hm_df = hm_df,  title_patient="Pt5", sketchy_flag=0)

In [ ]:
plot_hla_heatmap(hm_df = hm_df, title_patient="Pt8", sketchy_flag=0)

In [ ]:
plot_hla_heatmap(hm_df = hm_df, title_patient="Pt11", sketchy_flag=0)

In [ ]:
plot_hla_heatmap(hm_df = hm_df,  title_patient="Pt12", sketchy_flag=0)

In [ ]:
plot_hla_heatmap(hm_df = hm_df,  title_patient="Pt14", sketchy_flag=0)

In [ ]:
plot_hla_heatmap(hm_df = hm_df,  title_patient="Pt17")



In [ ]:
plot_hla_heatmap(hm_df = hm_df, title_patient="Pt18")

In [ ]:
plot_hla_heatmap(hm_df = hm_df, title_patient="Pt19")

In [ ]:
plot_hla_heatmap(hm_df = hm_df, title_patient="Pt20")

In [ ]:
plot_hla_heatmap(hm_df = hm_df, title_patient="Pt21")

# Display clones specific and non-specific for a single multimer
#### Step 1: Prepare a function to get the clone_order from the heatmap df

In [ ]:
# --------------------------------------------------
# 0) Define a dictionary for each donor linking donors to hla lists
# --------------------------------------------------
donor_hlas = {
    "Pt1" : ["A0201", "B0702", "C0702"],
    "Pt4" : ["B3501"],
    "Pt5" : ["A0101", "A2402", "B0801", "B1501"],
    "Pt8" : ["A0201", "B0702", "C0702"],
    "Pt11" : ["A0101", "B1501"], 
    "Pt12" : ["A0201"], 
    "Pt14" : ["A0201", "A6801", "B0702", "C0702"], 
    "Pt17" : ["A0101", "B1501", "B3501"], 
    "Pt18" : ["A0101", "A2402", "B0801"], 
    "Pt19" : ["A0201", "A2402", "B0702", "B1501", "C0702"], 
    "Pt20" : ["A0201", "A6801", "B0702", "B0801", "C0702"], 
    "Pt21" : ["A0201", "A2402", "B0702", "C0702"], 
}

hla_list = donor_hlas["Pt20"]

# --------------------------------------------------
# 1) Subset rows for selected HLAs
# --------------------------------------------------
hm_subset_df = hm_df[hm_df["HLA"].isin(donor_hlas["Pt20"])].copy()

# --------------------------------------------------
# 2) Select relevant columns
# --------------------------------------------------
manual_cols = ["donor", "clone_handle", "clone_size", "handle", "HLA"]

selected_cols = hm_subset_df.columns[
    hm_subset_df.columns.str.contains("|".join(hla_list))
]

final_cols = manual_cols + list(selected_cols)
hm_subset_df = hm_subset_df.loc[:, final_cols]

# --------------------------------------------------
# 3) Filter on clone size and patient
# --------------------------------------------------
hm_subset_df = hm_subset_df[
    (hm_subset_df["clone_size"] >= 2)
    & (hm_subset_df["donor"] == "Pt20")
]

# --------------------------------------------------
# 4) Identify epitope columns
# --------------------------------------------------
value_cols = hm_subset_df.columns[
    hm_subset_df.columns.str.contains(
        "|".join([f"_{h}_" for h in hla_list])
    )
]

# --------------------------------------------------
# 5) Build heatmap matrix
# --------------------------------------------------
heatmap_data = (
    hm_subset_df
    .set_index("clone_handle")[value_cols]
    .apply(pd.to_numeric, errors="coerce")
    .groupby(level=0)
    .mean()
    .T
)

# --------------------------------------------------
# 6) Ordering
# --------------------------------------------------

# Sort epitopes per HLA
epitope_order = sorted(
    heatmap_data.index,
    key=lambda x: x.split("_")[1]  # HLA position
)

# Sort each clone by the maximum epitope / multimer identity
# 1) epitope of max response (y-axis identity)
max_epitope = heatmap_data.idxmax(axis=0)

# 2) value of that max response
max_value = heatmap_data.max(axis=0)

# Combine into a DataFrame for sorting
clone_sort_df = pd.DataFrame({
    "clone": heatmap_data.columns,
    "max_epitope": max_epitope,
    "max_value": max_value,
})

# Sort:
#   - group by epitope
#   - within each epitope, sort by response strength
clone_sort_df = clone_sort_df.sort_values(
    by=["max_epitope", "max_value"],
    ascending=[True, False],
)

clone_order = clone_sort_df["clone"]

# implement orders. 
heatmap_data = heatmap_data.loc[epitope_order, clone_order]

# heatmap_data

In [ ]:
def get_clone_order(
    hm_df,  #The heatmap dataframe - do not change
    min_clone_size=2, 
    patient="Pt20", 
):
    # --------------------------------------------------
    # 0) Define a dictionary for each donor linking donors to hla lists
    # --------------------------------------------------
    donor_hlas = {
        "Pt1" : ["A0201", "B0702", "C0702"],
        "Pt4" : ["B3501"],
        "Pt5" : ["A0101", "A2402", "B0801", "B1501"],
        "Pt8" : ["A0201", "B0702", "C0702"],
        "Pt11" : ["A0101", "B1501"], 
        "Pt12" : ["A0201"], 
        "Pt14" : ["A0201", "A6801", "B0702", "C0702"], 
        "Pt17" : ["A0101", "B1501", "B3501"], 
        "Pt18" : ["A0101", "A2402", "B0801"], 
        "Pt19" : ["A0201", "A2402", "B0702", "B1501", "C0702"], 
        "Pt20" : ["A0201", "A6801", "B0702", "B0801", "C0702"], 
        "Pt21" : ["A0201", "A2402", "B0702", "C0702"], 
    }

    hla_list = donor_hlas[patient]
    
    # --------------------------------------------------
    # 1) Subset rows for selected HLAs
    # --------------------------------------------------
    hm_subset_df = hm_df[hm_df["HLA"].isin(hla_list)].copy()
    
    # --------------------------------------------------
    # 2) Select relevant columns
    # --------------------------------------------------
    manual_cols = ["donor", "clone_handle", "clone_size", "handle", "HLA"]
    
    selected_cols = hm_subset_df.columns[
        hm_subset_df.columns.str.contains("|".join(hla_list))
    ]
    
    final_cols = manual_cols + list(selected_cols)
    hm_subset_df = hm_subset_df.loc[:, final_cols]
    
    # --------------------------------------------------
    # 3) Filter on clone size and patient
    # --------------------------------------------------
    hm_subset_df = hm_subset_df[
        (hm_subset_df["clone_size"] >= min_clone_size)
        & (hm_subset_df["donor"] == patient)
    ]
    
    # --------------------------------------------------
    # 4) Identify epitope columns
    # --------------------------------------------------
    value_cols = hm_subset_df.columns[
        hm_subset_df.columns.str.contains(
            "|".join([f"_{h}_" for h in hla_list])
        )
    ]
    
    # --------------------------------------------------
    # 5) Build heatmap matrix
    # --------------------------------------------------
    heatmap_data = (
        hm_subset_df
        .set_index("clone_handle")[value_cols]
        .apply(pd.to_numeric, errors="coerce")
        .groupby(level=0)
        .mean()
        .T
    )
    
    # --------------------------------------------------
    # 6) Ordering
    # --------------------------------------------------
    
    # Sort epitopes per HLA
    epitope_order = sorted(
        heatmap_data.index,
        key=lambda x: x.split("_")[1]  # HLA position
    )
    
    # Sort each clone by the maximum epitope / multimer identity
    # 1) epitope of max response (y-axis identity)
    max_epitope = heatmap_data.idxmax(axis=0)
    
    # 2) value of that max response
    max_value = heatmap_data.max(axis=0)
    
    # Combine into a DataFrame for sorting
    clone_sort_df = pd.DataFrame({
        "clone": heatmap_data.columns,
        "max_epitope": max_epitope,
        "max_value": max_value,
    })
    
    # Sort:
    #   - group by epitope
    #   - within each epitope, sort by response strength
    clone_sort_df = clone_sort_df.sort_values(
        by=["max_epitope", "max_value"],
        ascending=[True, False],
    )
    
    clone_order = clone_sort_df["clone"]
    
    # implement orders. 
    heatmap_data = heatmap_data.loc[epitope_order, clone_order]
    
    return clone_order

### Step 2: prepare the dataframe from mdata
Subset on important columns. Rename value columns. Define a target_col and multimer dictionary, all target columns to the exact multimer name if there is a response in the dataset. 

In [ ]:
# --------------------
## For a dot plot we need log transformed values
dot_df = mdata["gex"].obs.copy()

# Suppose your dataframe is df
manual_cols = ["donor", "clone_handle", "HLA", "handle", "Credible_clones", "Virus", "Peptide", "Sequence","clone_size", "max_dextramer", "%_max_dextramer", "specific"]  # your manually selected columns

# Select columns that contain "log_sc-1OS"
selected_cols = dot_df.columns[dot_df.columns.str.contains("log_sc-1OS")]

# Combine with manual columns
final_cols =  manual_cols + list(selected_cols)

# Subset the dataframe
dot_df= dot_df[final_cols]
# --------------------


# --------------------
# Rename value columns:
epitope_ref = pd.read_excel("../../03_annotation/11_multimer_experiment_ann/FeatureBarcodCorrected.xlsx")
epitope_ref = epitope_ref.dropna()

#Define a column name mirroring the "log_XXX" columns
epitope_ref["targ_col"] = "log_" + epitope_ref["id"]
epitope_ref["handle"] = epitope_ref["Sequence"] + "_" + epitope_ref["HLA"] + "_" + epitope_ref["Virus"] 

# Create dictionary mapping targ_col -->  handle
col_dict = epitope_ref[["targ_col", "handle"]].set_index('targ_col')['handle'].to_dict()

# Rename columns where there is a match
dot_df = dot_df.rename(columns=col_dict)

# --------------------

# --------------------
# Only work with expanded clones. 
## This also gets rid of cells where the multimer bound - but no TCR was detected
dot_df = dot_df[dot_df["clone_size"] >= 2]

# --------------------

### Step 3: Prepare a function to plot CITEseq values across a clones of a given donor
Takes:
- The prepared data frame dot_df
- Donor
- multimer_target (the response you wish to highlight values for)
- min clone size
- a savepath

In [ ]:
def plot_multimer_across_expanded_clones(
    dot_df,
    donor,
    multimer_target,
    min_clone_size=2,
    figsize_scale=0.5,
    savepath=None,
):
    """
    Plot values for a target multimer across all expanded clones in a donor.

    Parameters
    ----------
    dot_df : pd.DataFrame
        Long-format dataframe containing clone, donor, multimer values
    donor : str
        Donor ID (e.g. "Pt4")
    multimer_target : str
        Column name of target multimer
        (e.g. "EPLPQGQLTAY_B3501_EBV")
    min_clone_size : int
        Minimum clone size to include
    figsize_scale : float
        Width scaling per clone
    savepath : str or Path or None
        Output PDF path
    """

    # --------------------------------------------------
    # 0) Get clone_order
    # --------------------------------------------------
    clone_order = get_clone_order(hm_df, patient=donor)
    
    # --------------------------------------------------
    # 1) Subset expanded clones for donor
    # --------------------------------------------------
    plot_df = dot_df[
        (dot_df["donor"] == donor) &
        (dot_df["clone_size"] >= min_clone_size) & 
        (dot_df["clone_handle"].isin(clone_order)
        # (~dot_df["clone_handle"].str.contains("nan")
        )
    ].copy()

    # Identify clones to display (ordered by expansion)
    clones_to_display = (
        plot_df[["clone_handle", "clone_size"]]
        .drop_duplicates()
        .sort_values("clone_size", ascending=False)
        ["clone_handle"]
        .tolist()
    )

    # Restrict dataframe to these clones
    plot_df = plot_df[
        plot_df["clone_handle"].isin(clones_to_display)
    ].copy()

    # --------------------------------------------------
    # 2) Clean categories
    # --------------------------------------------------
    if hasattr(plot_df["handle"], "cat"):
        plot_df["handle"] = plot_df["handle"].cat.remove_unused_categories()

    plot_df = plot_df.rename(
        columns={"handle": "Multimer assignment"}
    )

    # --------------------------------------------------
    # 3) Define specificity coloring
    # --------------------------------------------------
    plot_df["catcolor"] = np.where(
        plot_df["Multimer assignment"] == multimer_target,
        "Specific = True",
        "Specific = False",
    )

    catcolor_dic = {
        "Specific = True": "darkred",
        "Specific = False": "lightgrey",
    }

    # --------------------------------------------------
    # 4) Dynamic figure size
    # --------------------------------------------------
    fig_width = max(figsize_scale * len(clones_to_display), 4)

    # --------------------------------------------------
    # 5) Plot
    # --------------------------------------------------
    plt.figure(figsize=(fig_width, 3))

    ax = sb.stripplot(
        data=plot_df,
        x="clone_handle",
        y=multimer_target,
        order=clone_order,
        jitter=0.10,
        alpha=0.5,
        size=5,
        hue="catcolor",
        palette=catcolor_dic,
    )

    plt.ylabel("Log(UMI+1)")
    plt.xlabel("Clone")
    plt.xticks(rotation=45, ha="right")

    ax.legend(
        title="Multimer specificity",
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        borderaxespad=0,
        markerscale=3  # scales dot size in legend relative to plot
    )

    plt.title(f"{multimer_target} binding — {donor}")
    ax.set_ylim(-1,6.1)
    plt.tight_layout()

    # --------------------------------------------------
    # 6) Save
    # --------------------------------------------------
    if savepath is None:
        savepath = Path(
            f"../../05_plots/09_multimers/01_dotplots_perpatient/"
            f"Multimer_{multimer_target}_{donor}.pdf"
        )

    savepath = Path(savepath)
    savepath.parent.mkdir(parents=True, exist_ok=True)

    plt.savefig(savepath, format="pdf", dpi=400)
    plt.show()

    return clones_to_display

In [ ]:
## Multimer list
Pt1_list = dot_df[(dot_df["donor"] == "Pt1") & (dot_df["clone_size"] >= 2) & (~dot_df["handle"].isna())]["handle"].drop_duplicates().tolist() 
Pt1_list += ["LLIEGIFFI_A0201_EBV", "ILIEGVFFA_A0201_HHV-2"] #Extend the list
Pt4_list = dot_df[(dot_df["donor"] == "Pt4") & (dot_df["clone_size"] >= 2) & (~dot_df["handle"].isna())]["handle"].drop_duplicates().tolist()
Pt4_list += ["HPVAEADYFEY_B3501_EBV"]
Pt5_list = dot_df[(dot_df["donor"] == "Pt5") & (dot_df["clone_size"] >= 2) & (~dot_df["handle"].isna())]["handle"].drop_duplicates().tolist()
Pt8_list = dot_df[(dot_df["donor"] == "Pt8") & (dot_df["clone_size"] >= 2) & (~dot_df["handle"].isna())]["handle"].drop_duplicates().tolist()
Pt11_list = dot_df[(dot_df["donor"] == "Pt11") & (dot_df["clone_size"] >= 2) & (~dot_df["handle"].isna())]["handle"].drop_duplicates().tolist()
Pt12_list = dot_df[(dot_df["donor"] == "Pt12") & (dot_df["clone_size"] >= 2) & (~dot_df["handle"].isna())]["handle"].drop_duplicates().tolist()
Pt14_list = dot_df[(dot_df["donor"] == "Pt14") & (dot_df["clone_size"] >= 2) & (~dot_df["handle"].isna())]["handle"].drop_duplicates().tolist()
Pt17_list = dot_df[(dot_df["donor"] == "Pt17") & (dot_df["clone_size"] >= 2) & (~dot_df["handle"].isna())]["handle"].drop_duplicates().tolist()
Pt18_list = dot_df[(dot_df["donor"] == "Pt18") & (dot_df["clone_size"] >= 2) & (~dot_df["handle"].isna())]["handle"].drop_duplicates().tolist()
Pt19_list = dot_df[(dot_df["donor"] == "Pt19") & (dot_df["clone_size"] >= 2) & (~dot_df["handle"].isna())]["handle"].drop_duplicates().tolist()
Pt20_list = dot_df[(dot_df["donor"] == "Pt20") & (dot_df["clone_size"] >= 2) & (~dot_df["handle"].isna())]["handle"].drop_duplicates().tolist()
Pt21_list = dot_df[(dot_df["donor"] == "Pt21") & (dot_df["clone_size"] >= 2) & (~dot_df["handle"].isna())]["handle"].drop_duplicates().tolist()

In [ ]:
# -------
# Patient 1
# -------
print("Multimers with responses are: ", Pt1_list)

# Use the function together with a driver loop. 
for multimer in Pt1_list:
    plot_multimer_across_expanded_clones(
        dot_df,
        donor="Pt1",
        multimer_target=multimer,
        min_clone_size=2,
        figsize_scale=0.5,
    )

In [ ]:
# -------
# Patient 4
# -------
print("Multimers with responses are: ", Pt4_list)

# Use the function together with a driver loop. 
for multimer in Pt4_list:
    plot_multimer_across_expanded_clones(
        dot_df,
        donor="Pt4",
        multimer_target=multimer,
        min_clone_size=2,
        figsize_scale=0.2,
    )

In [ ]:
# -------
# Patient 5
# -------
print("Multimers with responses are: ", Pt5_list)

# Use the function together with a driver loop. 
for multimer in Pt5_list:
    plot_multimer_across_expanded_clones(
        dot_df,
        donor="Pt5",
        multimer_target=multimer,
        min_clone_size=2,
        figsize_scale=0.25,
    )

In [ ]:
# -------
# Patient 11
# -------
print("Multimers with responses are: ", Pt11_list)

# Use the function together with a driver loop. 
for multimer in Pt11_list:
    plot_multimer_across_expanded_clones(
        dot_df,
        donor="Pt11",
        multimer_target=multimer,
        min_clone_size=2,
        figsize_scale=0.25,
    )

In [ ]:
# -------
# Patient 12
# -------
print("Multimers with responses are: ", Pt12_list)

# Use the function together with a driver loop. 
for multimer in Pt12_list:
    plot_multimer_across_expanded_clones(
        dot_df,
        donor="Pt12",
        multimer_target=multimer,
        min_clone_size=2,
        figsize_scale=0.25,
    )

In [ ]:
# -------
# Patient 14
# -------
print("Multimers with responses are: ", Pt14_list)

# Use the function together with a driver loop. 
for multimer in Pt14_list:
    plot_multimer_across_expanded_clones(
        dot_df,
        donor="Pt14",
        multimer_target=multimer,
        min_clone_size=2,
        figsize_scale=0.25,
    )

In [ ]:
# -------
# Patient 17
# -------
print("Multimers with responses are: ", Pt17_list)

# Use the function together with a driver loop. 
for multimer in Pt17_list:
    plot_multimer_across_expanded_clones(
        dot_df,
        donor="Pt17",
        multimer_target=multimer,
        min_clone_size=2,
        figsize_scale=0.25,
    )

In [ ]:
# -------
# Patient 18
# -------
print("Multimers with responses are: ", Pt18_list)

# Use the function together with a driver loop. 
for multimer in Pt18_list:
    plot_multimer_across_expanded_clones(
        dot_df,
        donor="Pt18",
        multimer_target=multimer,
        min_clone_size=2,
        figsize_scale=0.25,
    )

In [ ]:
# -------
# Patient 19
# -------
print("Multimers with responses are: ", Pt19_list)

# Use the function together with a driver loop. 
for multimer in Pt19_list:
    plot_multimer_across_expanded_clones(
        dot_df,
        donor="Pt19",
        multimer_target=multimer,
        min_clone_size=2,
        figsize_scale=0.25,
    )

In [ ]:
# -------
# Patient 20
# -------
print("Multimers with responses are: ", Pt20_list)

# Use the function together with a driver loop. 
for multimer in Pt20_list:
    plot_multimer_across_expanded_clones(
        dot_df,
        donor="Pt20",
        multimer_target=multimer,
        min_clone_size=2,
        figsize_scale=0.25,
    )

In [ ]:
# -------
# Patient 21
# -------
print("Multimers with responses are: ", Pt21_list)

# Use the function together with a driver loop. 
for multimer in Pt21_list:
    plot_multimer_across_expanded_clones(
        dot_df,
        donor="Pt21",
        multimer_target=multimer,
        min_clone_size=2,
        figsize_scale=0.75,
    )

# Further Annotation
### Step 1: Remove multimer annotations from responses with less than 2 cells

In [ ]:
mdata["gex"].obs["handle"] = pd.Categorical(np.where(mdata["gex"].obs["clone_size"] >=2, mdata["gex"].obs["handle"], pd.NA))
mdata["gex"].obs["Virus"] = pd.Categorical(np.where(mdata["gex"].obs["clone_size"] >=2, mdata["gex"].obs["Virus"], pd.NA))
mdata["gex"].obs["Sequence"] = pd.Categorical(np.where(mdata["gex"].obs["clone_size"] >=2, mdata["gex"].obs["Sequence"], pd.NA))
mdata["gex"].obs["HLA"] = pd.Categorical(np.where(mdata["gex"].obs["clone_size"] >=2, mdata["gex"].obs["HLA"], pd.NA))

### Step 2: Annotate with EBV and CMV serology

In [ ]:
## Which donors are seronegative?
Patients = ["Pt1", "Pt4", "Pt5", "Pt8", "Pt11", "Pt12", "Pt14", "Pt17", "Pt18", "Pt19", "Pt20", "Pt21"] 
CMVsero = ["NA", "Negative", "Negative", "Negative", "Negative", "Positive", "Positive", "Negative", "Positive", "Positive", "Negative", "Negative"]
EBVsero = ["NA", "Positive", "Positive", "Positive", "Positive", "Positive", "Positive", "Positive", "Positive", "Positive", "Positive", "Positive"]

# initialize data of lists.
data = {'Donor':Patients,
        'CMVsero':CMVsero,
       "EBVsero" :EBVsero}

# Create DataFrame
df = pd.DataFrame(data)

# Create donor -> CMV serostatus dictionary
cmv_map = dict(zip(df["Donor"], df["CMVsero"]))
ebv_map = dict(zip(df["Donor"], df["EBVsero"]))

# Annotate using map! 
mdata["gex"].obs["CMVsero"] = mdata["gex"].obs["donor"].map(cmv_map).astype("category")
mdata["gex"].obs["EBVsero"] = mdata["gex"].obs["donor"].map(ebv_map).astype("category")

# Check
mdata["gex"].obs["CMVsero"].drop_duplicates().tolist()  #Ok 
mdata["gex"].obs[mdata["gex"].obs["CMVsero"].isna()]["donor"].drop_duplicates()  #ok 
mdata["gex"].obs[mdata["gex"].obs["CMVsero"]=="NA"]["donor"].drop_duplicates()  #ok 
mdata["gex"].obs[mdata["gex"].obs["CMVsero"]=="Positive"][["donor", "catpat"]].drop_duplicates()  #ok - 2 patients are CMV positive and 2 controls are CMV positive

# Check EBVsero
mdata["gex"].obs[mdata["gex"].obs["EBVsero"]=="Positive"][["donor", "catpat"]].drop_duplicates()  #ok - All patients are positive. 

## Step 3: Annotate CMV responses in CMV negative donors

In [ ]:
## Compare the Virus column with the CMVsero column
mdata["gex"].obs["CMV_sero_resp_eval"] = pd.Categorical(np.where((mdata["gex"].obs["Virus"] == "CMV") & (mdata["gex"].obs["CMVsero"] == "Negative"), "Seronegative, CMV-specific T cells",
                                                                         np.where((mdata["gex"].obs["Virus"] == "CMV") & (mdata["gex"].obs["CMVsero"] == "Positive"),  "Seropositive, CMV-specific T cells", "Other")))

mdata["gex"].obs[["handle", "CMV_sero_resp_eval", "donor"]].value_counts() 

In [ ]:
mdata["gex"].obs[["handle", "clone_handle"]].drop_duplicates()["handle"].value_counts() 

# Export mdata

In [ ]:
mdata.write(filename="../../04_data_objects/06_preprocessed_v2/2026_6_multimer_dataset_preprocessed_annotated.h5mu")